In [ ]:
import os
import sys
import django
PATH_PROJECT = fr"D:\PROGRAMMING\Projects\siakad\siakad_django"
sys.path.append(PATH_PROJECT)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'siakad_django.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()
from django.db import transaction

from django.db.models import Prefetch, Q

from django.conf import settings

In [ ]:
import pandas as pd

buat krs mahasiswa

In [ ]:
df = pd.read_clipboard()

In [ ]:
df

In [ ]:
from accounts.models import Mahasiswa, TandaTangan

In [ ]:
df.tanggal_lahir

In [ ]:
print("Total data       :", len(df))
print("Tanggal valid    :", df.tanggal_lahir.notna().sum())
print("Tanggal kosong/tidak valid:", df.tanggal_lahir.isna().sum())

In [ ]:
mahasiswa_26_qs = (
    Mahasiswa.objects
    .filter(
        nim__startswith="26"
    )
)
mahasiswa_26_qs
len(mahasiswa_26_qs)

In [ ]:
from django.contrib.auth.models import User
# make password
from django.contrib.auth.hashers import make_password

In [ ]:
list_user = []

for mahasiswa in mahasiswa_26_qs:
    temp = User(
        username=mahasiswa.nim,
        first_name=mahasiswa.nama,
    )

    temp.set_password(mahasiswa.nim)
    list_user.append(temp)

User.objects.bulk_create(list_user)

In [84]:
from django.contrib.auth.models import User
from django.db import transaction

with transaction.atomic():
    list_user = []

    for mahasiswa in mahasiswa_26_qs:
        temp = User(
            username=mahasiswa.nim,
            first_name=mahasiswa.nama,
        )
        temp.set_password(mahasiswa.nim)
        list_user.append(temp)

    # Simpan seluruh user sekaligus
    User.objects.bulk_create(list_user)

    # Ambil kembali user berdasarkan username (NIM)
    user_map = {
        user.username: user
        for user in User.objects.filter(
            username__in=[m.nim for m in mahasiswa_26_qs]
        )
    }

    # Hubungkan user ke mahasiswa
    for mahasiswa in mahasiswa_26_qs:
        user = user_map.get(mahasiswa.nim)

        if user:
            mahasiswa.user = user

    # Simpan relasi sekaligus
    mahasiswa_26_qs.model.objects.bulk_update(
        list(mahasiswa_26_qs),
        ["user"],
    )

In [85]:
tanda_tangan_list = []

for mahasiswa in mahasiswa_26_qs:
    if not mahasiswa.user_id:
        continue

    temp = TandaTangan(
        tanda_tangan=f"{mahasiswa.nim}.png",
        user=mahasiswa.user,
    )
    tanda_tangan_list.append(temp)

TandaTangan.objects.bulk_create(tanda_tangan_list)

[<TandaTangan: 167 - 26650033>,
 <TandaTangan: 168 - 26650035>,
 <TandaTangan: 169 - 26650055>,
 <TandaTangan: 170 - 26650063>,
 <TandaTangan: 171 - 26650064>,
 <TandaTangan: 172 - 26650065>,
 <TandaTangan: 173 - 26650066>,
 <TandaTangan: 174 - 26650067>,
 <TandaTangan: 175 - 26650083>,
 <TandaTangan: 176 - 26650084>,
 <TandaTangan: 177 - 26650085>,
 <TandaTangan: 178 - 26650088>,
 <TandaTangan: 179 - 26650090>,
 <TandaTangan: 180 - 26650097>,
 <TandaTangan: 181 - 26650098>,
 <TandaTangan: 182 - 26650114>,
 <TandaTangan: 183 - 26650115>,
 <TandaTangan: 184 - 26650116>,
 <TandaTangan: 185 - 26650117>,
 <TandaTangan: 186 - 26650118>,
 <TandaTangan: 187 - 26650120>,
 <TandaTangan: 188 - 26650122>,
 <TandaTangan: 189 - 26650123>,
 <TandaTangan: 190 - 26650125>,
 <TandaTangan: 191 - 26650128>,
 <TandaTangan: 192 - 26650129>,
 <TandaTangan: 193 - 26650131>,
 <TandaTangan: 194 - 26650133>,
 <TandaTangan: 195 - 26650134>,
 <TandaTangan: 196 - 26650135>,
 <TandaTangan: 197 - 26650137>,
 <TandaT

In [ ]:
TandaTangan.objects.filter(user=krs.mahasiswa.user).first()

In [136]:
nim = "24650123"
nim = "26650033"
mahasiswa = Mahasiswa.objects.filter(nim=nim).first()
mahasiswa
ttd = TandaTangan.objects.filter(
    user__mahasiswa=mahasiswa
).first()
ttd.tanda_tangan.path
# ttd.tanda_tangan

'D:\\PROGRAMMING\\Projects\\siakad\\siakad_django\\media\\tanda_tangan\\2026\\26650033.png'

In [112]:
ttd_path = settings.MEDIA_ROOT / "tanda_tangan"
ttd_path.exists()

True

In [117]:
ttd.tanda_tangan

WindowsPath('D:/PROGRAMMING/Projects/siakad/siakad_django/media/tanda_tangan/26650033.png')

In [149]:
mahasiswa_angkatan_24 = (
    Mahasiswa.objects
    .filter(
        nim__startswith="24"
    )
)
mahasiswa_angkatan_24
len(mahasiswa_angkatan_24)

mahasiswa_angkatan_22 = (
    Mahasiswa.objects
    .filter(
        nim__startswith="22"
    )
)
mahasiswa_angkatan_22
len(mahasiswa_angkatan_22)

33

In [150]:
tanda_tangan_list = []

qs = mahasiswa_angkatan_22
with transaction.atomic():
    for mahasiswa in qs:
        ttd = TandaTangan.objects.filter(
            user__mahasiswa=mahasiswa
        ).first()
        # tanda_tangan_list.append(temp)
        # update field
        angkatan = f"20{mahasiswa.nim[:2]}"
        ttd.tanda_tangan = f"tanda_tangan/{angkatan}/{mahasiswa.nim}.png"
        ttd.save()
    
tanda_tangan_list
len(tanda_tangan_list)

for value in tanda_tangan_list:
    print(value.tanda_tangan.path)

In [146]:
krs_id = 330
TandaTangan.objects.filter(
    user__mahasiswa__krs__id=krs_id
).first()

<TandaTangan: 206 - 26650170>

In [ ]:
"AGUS" in df.nama.values

In [ ]:
with transaction.atomic():
    hitung = 0
    for mahasiswa in mahasiswa_26_qs:
        if mahasiswa.nama in df.nama.values:
            mahasiswa.tanggal_lahir = ....
            hitung += 1
    print(hitung)

In [ ]:
with transaction.atomic():
    hitung = 0
    nama_tidak_cocok = []

    # Buat lookup berdasarkan NIM
    df_lookup = df.drop_duplicates(subset=["nim"]).set_index("nim")

    for mahasiswa in mahasiswa_26_qs:
        data = df_lookup.loc[mahasiswa.nim] if mahasiswa.nim in df_lookup.index else None

        if data is None:
            continue

        # Validasi nama untuk mencegah data yang keliru
        if str(mahasiswa.nama).strip().casefold() != str(data["nama"]).strip().casefold():
            nama_tidak_cocok.append(mahasiswa.nim)
            continue

        tanggal_lahir = data["tanggal_lahir"]

        # Jangan timpa tanggal lama jika tanggal dari Excel kosong/tidak valid
        if pd.isna(tanggal_lahir):
            continue

        mahasiswa.tanggal_lahir = tanggal_lahir.date()
        mahasiswa.save(update_fields=["tanggal_lahir"])
        hitung += 1

    print("Berhasil diperbarui:", hitung)
    print("NIM dengan nama tidak cocok:", nama_tidak_cocok)

In [ ]:
with transaction.atomic():
    hitung = 0
    nama_tidak_ditemukan = []
    nama_duplikat = []
    tanggal_kosong = []

    # Normalisasi nama agar tidak terpengaruh huruf besar/kecil dan spasi
    df_temp = df.copy()
    df_temp["nama_key"] = (
        df_temp["nama"].astype("string").str.strip().str.casefold()
    )

    for mahasiswa in mahasiswa_26_qs:
        nama_key = str(mahasiswa.nama).strip().casefold()

        data = df_temp.loc[df_temp["nama_key"] == nama_key]

        if data.empty:
            nama_tidak_ditemukan.append(mahasiswa.nama)
            continue

        if len(data) > 1:
            nama_duplikat.append(mahasiswa.nama)
            continue

        tanggal_lahir = data.iloc[0]["tanggal_lahir"]

        if pd.isna(tanggal_lahir):
            tanggal_kosong.append(mahasiswa.nama)
            continue

        mahasiswa.tanggal_lahir = pd.Timestamp(tanggal_lahir).date()
        mahasiswa.save(update_fields=["tanggal_lahir"])
        hitung += 1

    print("Berhasil diperbarui:", hitung)
    print("Nama tidak ditemukan:", nama_tidak_ditemukan)
    print("Nama duplikat di Excel:", nama_duplikat)
    print("Tanggal lahir kosong:", tanggal_kosong)

In [ ]:
# YYYY-MM-DD
df.tanggal_lahir = pd.to_datetime(
    df.tanggal_lahir,
    format="%Y-%m-%d",
    # errors="coerce"
)

In [ ]:
mahasiswa_list = []

for key, value in df.iterrows():
    temp = Mahasiswa(
        nim=value.nim,
        nama=value.nama,
        tempat_lahir=value.tempat_lahir,
        no_hp=value.no_hp,
        tanggal_lahir=(
            None if pd.isna(value.tanggal_lahir)
            else value.tanggal_lahir
        ),
        status="Aktif",
    )

    mahasiswa_list.append(temp)

In [ ]:
# with transaction.atomic():
#     Mahasiswa.objects.bulk_create(mahasiswa_list)
#     print("berhasil")

In [ ]:
mahasiswa_angkatan_26 = (
    Mahasiswa.objects
    .filter(
        nim__startswith="26"
    )
)
mahasiswa_angkatan_26
len(mahasiswa_angkatan_26)

In [ ]:
from academic.models import KRS, StatusKRS, PeriodeKRS

In [ ]:
periode_krs = PeriodeKRS.objects.filter(
    status__iexact="aktif"
).first()
periode_krs

In [ ]:
krs_list = []
for mahasiswa in mahasiswa_angkatan_26:
    krs_list.append(
        KRS(
            mahasiswa=mahasiswa,
            tahun_ajaran=periode_krs.tahun_ajaran,
            semester=periode_krs.semester,
            status=StatusKRS.SELESAI
        )
    )

In [ ]:
df

In [ ]:
with transaction.atomic():
    KRS.objects.bulk_create(krs_list)

In [ ]:
len(df)

In [ ]:
from academic.models import (
    KRS,
    DetailKRS, 
    PeriodeKRS,
    Mahasiswa,
    MataKuliah,
    Kelas,
    PenawaranKRS,
    StatusKRS
)

services

In [ ]:
from academic.krs.services import lihatDaftarKRS
from academic.krs.services.daftarHadir.daftarHadirService import (
    ExcelService, Query
)

In [ ]:
kelas_id = 130
template_path = settings.MEDIA_ROOT / "krs" / "template" / "DAFTAR_HADIR_TEMPLATE.xlsx"
dosen_nama = "Dodiman, S.Kom., M.Kom."
dosen_nidn = "0928079403"
hasil = ExcelService.excecute(
    kelas_id=kelas_id, template_path=template_path,
    DOSEN_PEMBINA=dosen_nama,
    NIDN=dosen_nidn
)
hasil.data

In [ ]:
with open("hasil.xlsx", "wb") as file:
    file.write(hasil.data.getvalue())

In [ ]:
svc = Query()
svc.mydata(kelas_id=130)['daftar_mahasiswa']
len(svc.mydata(kelas_id=130)['daftar_mahasiswa'])
svc.mydata(kelas_id=130)['mata_kuliah']
svc.mydata(kelas_id=130)['krs']
svc.mydata(kelas_id=130)['krs'].semester

In [ ]:
periode_krs = PeriodeKRS.objects.filter(status__iexact="aktif").first()
periode_krs

krs_id = 147
krs = KRS.objects.filter(id=krs_id).first()
krs

In [ ]:
svc = lihatDaftarKRS.Dataku()
res = svc.daftar_krs_by_kelas(periode_krs=periode_krs, kelas_id=130)
res
# len(res)
res[0]

penawaran_krs = PenawaranKRS.objects.filter(kelas__id=130).first()
kelas = penawaran_krs.kelas
periode_krs = penawaran_krs.periode_krs

kelas = Kelas.objects.get(id=130)
kelas.mata_kuliah
kelas.mata_kuliah.nama
kelas.mata_kuliah

periode_krs.id
periode_krs.tahun_ajaran
periode_krs.semester

openpyxl

In [ ]:
from pathlib import Path
import io

from openpyxl import Workbook, load_workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.utils import column_index_from_string
from openpyxl.utils.cell import coordinate_to_tuple

In [ ]:
template_daftar_hadir = settings.MEDIA_ROOT / "krs" / "template" / "DAFTAR_HADIR_TEMPLATE.xlsx"
template_daftar_hadir.exists()

In [ ]:
# path_file = fr"D:\PROGRAMMING\Projects\siakad\siakad_django\academic\krs\services\daftarHadir"
# template_daftar_hadir = Path(path_file) / "DAFTAR_HADIR_TEMPLATE.xlsx"
# template_daftar_hadir.exists()

In [ ]:
wb = load_workbook(template_daftar_hadir, data_only=True)
ws_daftar_hadir = wb["KELAS"]
ws_jurnal_perkuliahan = wb["JURNAL PERKULIAHAN"]
print("")

In [ ]:
class ExcelService:
    def __init__(self):
        pass
    
    @staticmethod
    def simpan_excel(wb) -> io.BytesIO:
        output = io.BytesIO()
        wb.save(output)
        output.seek(0)
        return output
    
    @staticmethod
    def sisipkan_array(ws, *, kolom: str, data: list, baris_awal: int = 1):
        nomor_kolom = column_index_from_string(kolom)

        for row, value in enumerate(data, start=baris_awal):
            ws.cell(row=row, column=nomor_kolom, value=value)
            
    @staticmethod
    def sisipkan_nilai_by_coordinat(ws, *, coordinate: str, value:str):
        baris, kolom = coordinate_to_tuple(coordinate=coordinate)
        
        ws.cell(row=baris, column=kolom, value=value)

In [ ]:
fieldSet_daftar_hadir = {
    "MATA_KULIAH_SKS":  "E8",
    "SEMESTER_RUANG": "E9",
    "DOSEN_PEMBINA": "E10",
    "NIDN": "L10",
    "WAKTU": "E11",
    "NAMA_KELAS": "S8",
    "NIM": "B17",  # cell pertama
    "NAMA": "C17", # cell pertama
    "DOSEN_PEMBINA_NAMA": "O65",
    "DOSEN_PEMBINA_NIDN": "P66",
}

fieldSet_jurnal_perkuliahan = {
    "TAHUN_AKADEMIK": "E8",
    "MATA_KULIAH_SKS": "E9",
    "SEMESTER": "E10",
    "DOSEN_PEMBINA": "E11"
}

In [ ]:
output_bytesIO = ExcelService.simpan_excel(wb)
with open("hasil.xlsx", "wb") as file:
    file.write(output_bytesIO.getvalue())

query

In [ ]:
krs_qs = (
    KRS.objects
    .filter(
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
    )
    .select_related("mahasiswa")
    # .values_list("id", flat=True)
)
krs_qs

In [ ]:
krs_by_mahasiswa = {
    krs.mahasiswa.id: krs
    for krs in krs_qs
}
krs_by_mahasiswa
set(krs_by_mahasiswa)


In [ ]:
list_krs_ids = [65, 67, 95, 119, 161]
krs_temp = KRS.objects.filter(id__in=list_krs_ids).values_list("status")
krs_temp

In [ ]:
# semua mahasiswa aktif yang telah mengisi krs

mahasiswa_qs = (
        Mahasiswa.objects.filter(
        krs__semester=periode_krs.semester,
        krs__tahun_ajaran=periode_krs.tahun_ajaran,
        status__iexact="aktif",
    )
    .values_list("id", flat=True)
)
mahasiswa_qs
len(mahasiswa_qs)

In [ ]:
# tambahkan kelas
x = 24
krs_angkatan_x = (
    KRS.objects
    .filter(
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
        mahasiswa__nim__startswith=x
    )
    # .values_list("mahasiswa__nim", flat=True)
    
)
krs_angkatan_x
# len(krs_angkatan_x)


In [ ]:
# tambahkan kelas
x = 24
krs_angkatan_x = (
    KRS.objects
    .filter(
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
        mahasiswa__nim__startswith=x
    )
    .values_list("id", flat=True)
    
)
krs_angkatan_x
len(krs_angkatan_x)


In [ ]:
kelas_ids = [
    127, 144

]

In [ ]:
from itertools import product

krs_ids = krs_angkatan_x
kelas_ids = kelas_ids
krs_ids
list(product(krs_ids, kelas_ids))

In [ ]:
detail_krs_qs = (
    DetailKRS.objects
    .filter(krs__id__in=krs_angkatan_x)
    # .select_related("kelas", "kelas__mata_kuliah")
    # .delete()
)
detail_krs_qs
len(detail_krs_qs)

In [ ]:
detail_krs_qs

In [ ]:
mahasiswa_belum_krs = (
    Mahasiswa.objects
    .filter(
    #     krs__semester=periode_krs.semester,
    #     krs__tahun_ajaran=periode_krs.tahun_ajaran,
        status__iexact="aktif",
    )
    .exclude(id__in=mahasiswa_qs)
    .values_list("id", flat=True)
)
mahasiswa_belum_krs
len(mahasiswa_belum_krs)

In [ ]:
# buat krs otomatis
krs_list = [
    KRS(
        mahasiswa_id=mhs_id,
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
        status=StatusKRS.DISETUJUI
    )
    for mhs_id in mahasiswa_belum_krs
]
krs_list
# len(krs_list)

with transaction.atomic():
    KRS.objects.bulk_create(krs_list)
    print("berhasil")

In [ ]:
from academic.krs.services.lihatDaftarKRS import serializers


In [ ]:
query_set = (
    MataKuliah.objects.filter(
        kelas__penawarankrs__periode_krs=periode_krs,
        kelas__peserta__krs__semester=periode_krs.semester,
        kelas__peserta__krs__tahun_ajaran=periode_krs.tahun_ajaran,
    )
    .distinct()
    .prefetch_related(Prefetch("kelas_set", to_attr="daftar_kelas"))
)

query_set
serializers.MataKuliahKelasSerializer(instance=query_set, many=True).data
# serializers.DetailKRSKKelasSerializer(instance=query_set, many=True).data

services

In [ ]:
from academic.krs.services.lihatDaftarKRS import LihatDaftarKRS
from academic.krs.services.lihatDaftarKRS import serializers

In [ ]:
qs = LihatDaftarKRS().krs().data
qs
serializers.KRSSerializer(instance=qs, many=True).data

In [ ]:
svc = LihatDaftarKRS()
svc.lihat_daftar_krs()

query

In [ ]:
kelas_id=127
hasil_qs = (
    KRS.objects
    .filter(
        semester = periode_krs.semester,
        tahun_ajaran = periode_krs.tahun_ajaran,
        detail__kelas__id = kelas_id
    )
    .select_related("mahasiswa")
)
hasil_qs
sr = serializers.KRSMahasiswaPesertaSerializer(instance=hasil_qs, many=True)
sr.data

In [ ]:
detail_krs_qs = (
    DetailKRS.objects
    .filter(
        krs__semester=periode_krs.semester,
        krs__tahun_ajaran=periode_krs.tahun_ajaran
    )
)
detail_krs_qs